# ADAS Vision — Phase 6: Learned Drivable-Area Segmentation

This notebook evaluates a **pretrained YOLOP model** (trained on BDD100K driving data) on our own dashcam footage. YOLOP does three things at once:

1. **Drivable-area segmentation** — pixel-level "where can the car go" (works on unmarked roads!)
2. **Lane-line segmentation** — pixel-level lane markings
3. **Vehicle detection**

**Goal:** see whether a learned model handles the scenes where classical CV struggled — unmarked hill roads, curves, mixed traffic — before spending anything on hardware (Jetson).

> **Safety note:** this project is advisory/research only. It must never be connected to a vehicle's steering, throttle or brakes.

## Setup (one-time)
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Upload your dashcam video to Google Drive (e.g. `MyDrive/adas/dashcam.mp4`)
3. Run the cells top to bottom

In [ ]:
# 1) Confirm we actually have a GPU (should show a Tesla T4)
!nvidia-smi -L
import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 2) Dependencies YOLOP needs beyond Colab's defaults
!pip -q install yacs prefetch_generator

In [ ]:
# 3) Load pretrained YOLOP from torch.hub (downloads weights on first run)
model = torch.hub.load("hustvl/yolop", "yolop", pretrained=True, trust_repo=True)
model = model.to(DEVICE).eval()
print("YOLOP loaded on", DEVICE)

In [ ]:
# 4) Mount Google Drive and point at the video
from google.colab import drive
drive.mount("/content/drive")

VIDEO_PATH = "/content/drive/MyDrive/adas/dashcam.mp4"   # <-- change if needed

import os
assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH} — upload the video to Drive and fix the path"
import cv2
cap = cv2.VideoCapture(VIDEO_PATH)
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); W = int(cap.get(3)); H = int(cap.get(4))
cap.release()
print(f"video: {W}x{H}, {N} frames")

In [ ]:
# 5) Inference helpers
import numpy as np
import torchvision.transforms as T

IN_W, IN_H = 640, 384    # YOLOP-friendly input (divisible by 32)
normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def infer(frame_bgr):
    """Run YOLOP on one BGR frame. Returns (drivable_mask, lane_mask) at frame size."""
    h, w = frame_bgr.shape[:2]
    img = cv2.resize(frame_bgr, (IN_W, IN_H))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    t = normalize(t).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        det_out, da_seg, ll_seg = model(t)
    da = da_seg.argmax(1).squeeze().detach().cpu().numpy().astype(np.uint8)
    ll = ll_seg.argmax(1).squeeze().detach().cpu().numpy().astype(np.uint8)
    da = cv2.resize(da, (w, h), interpolation=cv2.INTER_NEAREST)
    ll = cv2.resize(ll, (w, h), interpolation=cv2.INTER_NEAREST)
    return da, ll

def overlay(frame_bgr, da, ll):
    """Green = drivable area, red = lane lines."""
    out = frame_bgr.copy()
    color = np.zeros_like(out)
    color[da > 0] = (0, 180, 0)
    color[ll > 0] = (0, 0, 255)
    return cv2.addWeighted(out, 1.0, color, 0.45, 0)

print("helpers ready")

In [ ]:
# 6) THE KEY TEST — the exact scenes where classical CV failed.
#    Frame 15000 is the unmarked hill road that broke the Hough detector.
import matplotlib.pyplot as plt

TEST_FRAMES = [2000, 5000, 9000, 12000, 15000, 17000]

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(len(TEST_FRAMES), 1, figsize=(14, 7 * len(TEST_FRAMES)))
for ax, fi in zip(axes, TEST_FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ok, frame = cap.read()
    if not ok:
        continue
    da, ll = infer(frame)
    vis = overlay(frame, da, ll)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame {fi} — drivable area (green), lane lines (red)")
    ax.axis("off")
cap.release()
plt.tight_layout()
plt.show()

In [ ]:
# 7) Speed benchmark — GPU fps here, plus a rough CPU proxy for edge planning
import time

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 9000)
frames = []
for _ in range(60):
    ok, f = cap.read()
    if ok:
        frames.append(f)
cap.release()

# warmup
for f in frames[:5]:
    infer(f)
t0 = time.time()
for f in frames:
    infer(f)
gpu_fps = len(frames) / (time.time() - t0)
print(f"GPU ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'}): {gpu_fps:.1f} fps")
print("Note: a Jetson Orin Nano typically lands between laptop-CPU and T4 speeds;")
print("with TensorRT optimisation YOLOP runs real-time (>20 fps) on Orin-class boards.")

In [ ]:
# 8) Render an annotated video segment and save it to Drive
#    (start with the hill section — the one that broke classical CV)
from tqdm import tqdm

START, N_FRAMES = 14500, 900          # ~30s of the hill road
OUT_PATH = "/content/drive/MyDrive/adas/phase6_hills_segmented.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
fps_src = cap.get(cv2.CAP_PROP_FPS) or 30
cap.set(cv2.CAP_PROP_POS_FRAMES, START)
writer = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps_src, (W, H))
for _ in tqdm(range(N_FRAMES)):
    ok, frame = cap.read()
    if not ok:
        break
    da, ll = infer(frame)
    writer.write(overlay(frame, da, ll))
cap.release(); writer.release()
print("saved:", OUT_PATH)

In [ ]:
# 9) (Optional, for later) Export to ONNX — the format we'd deploy on a Jetson
#    or via OpenVINO on the laptop's Intel iGPU.
dummy = torch.randn(1, 3, IN_H, IN_W).to(DEVICE)
torch.onnx.export(
    model, dummy, "/content/drive/MyDrive/adas/yolop_640x384.onnx",
    input_names=["image"], output_names=["det", "drivable", "lane"],
    opset_version=12,
)
print("ONNX exported to Drive (adas/yolop_640x384.onnx)")

## What to look at

1. **Cell 6, frame 15000** — does the green drivable area sit on the unmarked hill road (where classical CV drew the 'X' on the hillside)? This is the whole reason for Phase 6.
2. **The rendered hill video (cell 8)** — watch whether the drivable area stays stable through the curves and around the cow.
3. **Where it fails** — note timestamps of any bad frames; those tell us whether the pretrained model is enough or needs fine-tuning on Indian-road data (e.g. the IDD dataset from IIIT Hyderabad).

## Next steps after this notebook

| Result | Next move |
|---|---|
| Drivable area is solid on our footage | Export ONNX → benchmark on laptop (OpenVINO) → decide Jetson |
| Struggles on Indian roads | Fine-tune on the [IDD dataset](https://idd.insaan.iiit.ac.in/) (also free on Colab) |

Either way, the model slots into the existing architecture as a drop-in guidance source: it produces the same `LaneDetectionResult` interface that `road_detection.py` uses today — the decision engine doesn't change at all.